# Program 7 – Word2Vec: Skip-Gram Model with Negative Sampling

**Objective:** Implement the Skip-Gram model to predict context words given a center word.

**Dataset:** Same small corpus as CBOW (Program 6)

**Key tasks**
1. Create training pairs (word, context).
2. Train the embedding layer using negative sampling.
3. Compare with CBOW embeddings.

## Theory (for viva)

**Skip-Gram:** given a **center** word, predict each word in its context window.
Sentence *"the king rules the kingdom"*, center = **rules**, window = 2 → pairs
(rules, the), (rules, king), (rules, the), (rules, kingdom).

**Problem:** a full softmax over the whole vocabulary for every pair is expensive.

**Negative Sampling (SGNS):** turn it into binary classification.
- **Positive pair** (center, real context word) → label 1.
- **K negative pairs** (center, random word from the vocabulary) → label 0.
- Score = dot product of the center-word vector and the context-word vector: $s = v_c \cdot u_o$.
- Loss for one example: $-\log \sigma(v_c \cdot u_o) - \sum_{k=1}^{K} \log \sigma(-v_c \cdot u_k)$

Negatives are sampled from the **unigram distribution raised to 3/4**, $P(w) \propto f(w)^{0.75}$, which gives rare words a slightly higher chance.

**Two embedding matrices:** *target* (center) embeddings $v$ and *context* embeddings $u$. After training, the target embedding matrix is used as the word vectors.

## Step 1 – Import libraries

In [ ]:
import keras  # Keras 3 API (runs on TensorFlow backend)
from keras import layers  # Layer classes (Dense, Dropout, Embedding, ...)
import matplotlib.pyplot as plt  # Plotting graphs
import numpy as np  # Numerical arrays
from sklearn.decomposition import PCA  # PCA for 2-D visualization
import tensorflow as tf  # TensorFlow (tensors, GradientTape, seeds)

tf.random.set_seed(42)                          # TensorFlow reproducibility
np.random.seed(42)                              # NumPy reproducibility

## Step 2 – Corpus (same as CBOW)

In [ ]:
corpus = [                                                                  # Small custom corpus: list of sentences
    "the king rules the kingdom with the queen",                            # Royalty theme
    "the queen is the wife of the king",                                    # Royalty theme
    "the prince is the son of the king and the queen",                      # Royalty theme
    "the princess is the daughter of the king and the queen",               # Royalty theme
    "a man can become a king and a woman can become a queen",               # Royalty + gender
    "the king and the prince ride horses in the kingdom",                   # Royalty theme
    "the queen and the princess live in the royal palace",                  # Royalty theme
    "the royal palace belongs to the king",                                 # Royalty theme
    "the man is the father and the woman is the mother",                    # Family theme
    "the boy is a young man and the girl is a young woman",                 # Family theme
    "the father and the mother love their son and daughter",                # Family theme
    "the dog chases the cat in the garden",                                 # Animals theme
    "the cat drinks milk and the dog eats meat",                            # Animals theme
    "a dog is a loyal animal and a cat is a clever animal",                 # Animals theme
    "the horse and the cow are farm animals",                               # Animals theme
    "the cow gives milk and the horse pulls the cart",                      # Animals theme
    "the puppy is a young dog and the kitten is a young cat",               # Animals theme
    "apple and banana are sweet fruits",                                    # Fruits theme
    "mango is the king of fruits",                                          # Fruits theme
    "she eats an apple and a banana every morning",                         # Fruits theme
    "orange and mango juice are healthy drinks",                            # Fruits theme
    "fresh fruits like apple orange and mango are healthy",                 # Fruits theme
    "the computer runs software written in python",                         # Tech theme
    "python is a popular programming language",                             # Tech theme
    "deep learning models run on a computer with a gpu",                    # Tech theme
    "neural networks learn patterns from data",                             # Tech theme
    "a programmer writes code in python on a computer",                     # Tech theme
    "the gpu makes deep learning training fast",                            # Tech theme
    "data is used to train neural networks and deep learning models",       # Tech theme
    "machine learning and deep learning need a lot of data",                # Tech theme
]
print("Number of sentences:", len(corpus))                                  # How many sentences we have

## Step 3 – Tokenize and build vocabulary (same order as CBOW, so indices match)

In [ ]:
tokenized = [sentence.lower().split() for sentence in corpus]               # Lowercase and split each sentence into words

all_words = sorted({w for sent in tokenized for w in sent})                 # Unique words, sorted for a fixed order
word2idx = {"<pad>": 0}                                                     # Index 0 reserved for padding
for w in all_words:                                                         # Give every real word an index from 1
    word2idx[w] = len(word2idx)                                             # Next free index
idx2word = {i: w for w, i in word2idx.items()}                              # Reverse lookup: index -> word
vocab_size = len(word2idx)                                                  # Total vocabulary size (incl. <pad>)

sequences = [[word2idx[w] for w in sent] for sent in tokenized]             # Convert each sentence to a list of indices

print("Vocabulary size:", vocab_size)                                       # Print vocab size
print("First sentence as indices:", sequences[0])                           # Example encoding

## Step 4 – Create positive (center, context) pairs

In [ ]:
WINDOW = 2                                                                  # Context words on each side

positive_pairs = []                                                         # List of (center_id, context_id)
for seq in sequences:                                                       # Loop over sentences
    for center_pos, center_word in enumerate(seq):                          # Every word acts as a center once
        for offset in range(-WINDOW, WINDOW + 1):                           # Offsets -2 .. 2
            ctx_pos = center_pos + offset                                   # Position of context word
            if offset == 0 or ctx_pos < 0 or ctx_pos >= len(seq):           # Skip center itself and out-of-range
                continue                                                    # Next offset
            positive_pairs.append((center_word, seq[ctx_pos]))              # Save positive pair

positive_pairs = np.array(positive_pairs, dtype="int32")                    # Shape (num_pairs, 2)
print("Positive pairs:", len(positive_pairs))                               # How many pairs
for c, o in positive_pairs[:8]:                                             # Show first 8 pairs
    print(f"({idx2word[c]}, {idx2word[o]})")                                # (center, context) as words

## Step 5 – Negative sampling distribution  $P(w) \propto f(w)^{0.75}$

In [ ]:
counts = np.zeros(vocab_size)                                               # Word frequency array
for seq in sequences:                                                       # Loop over sentences
    for w in seq:                                                           # Loop over words
        counts[w] += 1                                                      # Count occurrences
counts[0] = 0                                                               # Never sample <pad>

noise_dist = counts ** 0.75                                                 # Raise frequencies to 3/4 power
noise_dist = noise_dist / noise_dist.sum()                                  # Normalize into a probability distribution

top = np.argsort(counts)[::-1][:5]                                          # 5 most frequent words
for i in top:                                                               # Compare raw frequency vs sampling prob
    print(f"{idx2word[i]:>6}: freq={counts[i] / counts.sum():.3f}  sample_prob={noise_dist[i]:.3f}")  # Frequent words are down-weighted

## Step 6 – Build training examples: 1 positive + K negatives each

Each example: `target` = center id, `context` = [true context id, neg1, ..., negK], `label` = [1, 0, ..., 0].

In [ ]:
NUM_NEG = 4                                                                 # K = number of negative samples per positive pair

targets, contexts, labels = [], [], []                                      # Lists to fill
for center, true_ctx in positive_pairs:                                     # Loop over positive pairs
    negatives = []                                                          # Negative word ids for this pair
    while len(negatives) < NUM_NEG:                                         # Keep sampling until we have K
        neg = np.random.choice(vocab_size, p=noise_dist)                    # Draw a random word from noise distribution
        if neg != true_ctx and neg != center:                               # Reject if it is the real context or center
            negatives.append(neg)                                           # Accept as negative
    targets.append([center])                                                # Target input (shape 1)
    contexts.append([true_ctx] + negatives)                                 # Context input: positive first, then negatives
    labels.append([1] + [0] * NUM_NEG)                                      # Label 1 for positive, 0 for negatives

targets = np.array(targets, dtype="int32")                                  # Shape (N, 1)
contexts = np.array(contexts, dtype="int32")                                # Shape (N, K+1)
labels = np.array(labels, dtype="float32")                                  # Shape (N, K+1)

print("targets:", targets.shape, "contexts:", contexts.shape, "labels:", labels.shape)  # Check shapes
print("Example -> center:", idx2word[targets[0, 0]],                        # First example center word
      "| candidates:", [idx2word[i] for i in contexts[0]],                  # Its candidate context words
      "| labels:", labels[0])                                               # Which one is real

## Step 7 – Build the Skip-Gram model (two embeddings + dot product)

In [ ]:
EMBED_DIM = 50                                                              # Same dimension as CBOW for fair comparison

target_input = keras.Input(shape=(1,), dtype="int32", name="target")        # Center word id
context_input = keras.Input(shape=(NUM_NEG + 1,), dtype="int32", name="context")  # 1 positive + K negative ids

target_emb = layers.Embedding(vocab_size, EMBED_DIM,                        # Target (center) embedding matrix v
                              name="target_embedding")(target_input)        # Output shape (batch, 1, 50)
context_emb = layers.Embedding(vocab_size, EMBED_DIM,                       # Context embedding matrix u
                               name="context_embedding")(context_input)     # Output shape (batch, K+1, 50)

dots = layers.Dot(axes=(2, 2))([context_emb, target_emb])                   # Dot product v·u for each candidate -> (batch, K+1, 1)
logits = layers.Flatten()(dots)                                             # Remove last axis -> (batch, K+1) raw scores

skipgram_model = keras.Model(inputs=[target_input, context_input],          # Model with two inputs
                             outputs=logits, name="skipgram_neg_sampling")  # Output: one score per candidate

skipgram_model.compile(                                                     # Configure training
    optimizer=keras.optimizers.Adam(learning_rate=0.01),                    # Adam optimizer
    loss=keras.losses.BinaryCrossentropy(from_logits=True),                 # Sigmoid + BCE on each score = negative-sampling loss
    metrics=[keras.metrics.BinaryAccuracy(threshold=0.0)],                  # Logit > 0 means "predicted real pair"
)
skipgram_model.summary()                                                    # Show architecture

## Step 8 – Train

In [ ]:
history = skipgram_model.fit(                                               # Train Skip-Gram
    {"target": targets, "context": contexts},                               # Inputs matched by layer name
    labels,                                                                 # 1/0 labels
    epochs=50,                                                              # Epochs
    batch_size=64,                                                          # Batch size
    shuffle=True,                                                           # Shuffle examples each epoch
    verbose=0,                                                              # Silent
)

plt.figure(figsize=(8, 4))                                                  # Figure
plt.plot(history.history["loss"])                                           # Loss curve
plt.title("Skip-Gram (Negative Sampling) Training Loss")                    # Title
plt.xlabel("Epoch")                                                         # X label
plt.ylabel("Binary cross-entropy")                                          # Y label
plt.grid(True)                                                              # Grid
plt.show()                                                                  # Show
print("Final loss:", history.history["loss"][-1])                           # Final loss value

## Step 9 – Extract Skip-Gram embeddings and find similar words

In [ ]:
sg_embeddings = skipgram_model.get_layer("target_embedding").get_weights()[0]  # Target embedding matrix = word vectors
print("Skip-Gram embedding shape:", sg_embeddings.shape)                    # (vocab_size, 50)


def most_similar(word, emb, top_n=5):                                       # Cosine-similarity nearest neighbours
    norm = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9)        # Unit-length vectors
    sims = norm @ norm[word2idx[word]]                                      # Cosine similarity to query word
    order = [i for i in np.argsort(sims)[::-1]                              # Indices sorted by similarity
             if i != word2idx[word] and i != 0]                             # Exclude the word itself and <pad>
    return [(idx2word[i], round(float(sims[i]), 3)) for i in order[:top_n]] # Top-n (word, similarity)

for w in ["king", "queen", "dog", "apple", "python"]:                       # Query words
    print(f"{w:>7} ->", most_similar(w, sg_embeddings))                     # Print neighbours

## Step 10 – Get CBOW embeddings for comparison

Loads `cbow_embeddings.npy` saved by Program 6 if it exists; otherwise quickly trains a CBOW model here on the same corpus.

In [ ]:
try:                                                                        # Did Program 6 save embeddings?
    cbow_embeddings = np.load("cbow_embeddings.npy")                        # Load them (fails if file missing)
    print("Loaded CBOW embeddings from Program 6")                          # Info
except FileNotFoundError:                                                   # Not found -> train a quick CBOW here
    ctx_list, center_list = [], []                                          # CBOW inputs / labels
    for seq in sequences:                                                   # Loop over sentences
        for pos, center in enumerate(seq):                                  # Each word as center
            ctx = [seq[pos + o] if 0 <= pos + o < len(seq) else 0           # Context id or <pad>
                   for o in range(-WINDOW, WINDOW + 1) if o != 0]           # Offsets except 0
            ctx_list.append(ctx)                                            # Save context
            center_list.append(center)                                      # Save center
    cbow = keras.Sequential([                                               # Same CBOW as Program 6
        keras.Input(shape=(2 * WINDOW,), dtype="int32"),                    # 4 context ids
        layers.Embedding(vocab_size, EMBED_DIM, mask_zero=True, name="cbow_embedding"),  # Embeddings
        layers.GlobalAveragePooling1D(),                                    # Average context vectors
        layers.Dense(vocab_size, activation="softmax"),                     # Predict center word
    ])
    cbow.compile(optimizer=keras.optimizers.Adam(0.01),                     # Adam
                 loss="sparse_categorical_crossentropy")                    # Integer labels
    cbow.fit(np.array(ctx_list), np.array(center_list),                     # Train
             epochs=200, batch_size=32, verbose=0)                          # 200 silent epochs
    cbow_embeddings = cbow.get_layer("cbow_embedding").get_weights()[0]     # Extract embeddings
    print("Trained CBOW inside this notebook")                              # Info

assert cbow_embeddings.shape == sg_embeddings.shape                         # Both must have same vocab & dim

## Step 11 – Compare nearest neighbours: CBOW vs Skip-Gram

In [ ]:
query_words = ["king", "queen", "prince", "dog", "cat", "apple", "mango", "python", "gpu"]  # Words to compare

print(f"{'word':>8} | {'CBOW neighbours':<45} | Skip-Gram neighbours")    # Table header
print("-" * 110)                                                            # Separator
for w in query_words:                                                       # One row per query word
    cb = ", ".join(x for x, _ in most_similar(w, cbow_embeddings, 4))       # CBOW top-4 words
    sg = ", ".join(x for x, _ in most_similar(w, sg_embeddings, 4))         # Skip-Gram top-4 words
    print(f"{w:>8} | {cb:<45} | {sg}")                                      # Print row

## Step 12 – Compare using theme-based similarity scores

In [ ]:
def cosine(a, b):                                                           # Cosine similarity of two vectors
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))  # a·b / (|a||b|)

test_pairs = [("king", "queen"), ("prince", "princess"), ("dog", "cat"),    # Pairs that SHOULD be similar
              ("apple", "mango"), ("python", "computer"),                   # (same theme)
              ("king", "apple"), ("dog", "python")]                         # Pairs that should NOT be similar

print(f"{'pair':<22} {'CBOW':>7} {'SkipGram':>9}")                          # Header
for a, b in test_pairs:                                                     # Loop over pairs
    c1 = cosine(cbow_embeddings[word2idx[a]], cbow_embeddings[word2idx[b]]) # CBOW similarity
    c2 = cosine(sg_embeddings[word2idx[a]], sg_embeddings[word2idx[b]])     # Skip-Gram similarity
    print(f"{a + ' / ' + b:<22} {c1:>7.3f} {c2:>9.3f}")                     # Print both

## Step 13 – Side-by-side PCA plots

In [ ]:
highlight = ["king", "queen", "prince", "princess", "royal", "palace",      # Royalty words
             "dog", "cat", "horse", "cow", "puppy", "kitten",               # Animal words
             "apple", "banana", "mango", "orange", "fruits",                # Fruit words
             "python", "computer", "gpu", "data", "learning", "neural"]     # Tech words
colors = (["red"] * 6 + ["green"] * 6 + ["orange"] * 5 + ["blue"] * 6)      # One colour per theme
ids = [word2idx[w] for w in highlight]                                      # Indices of these words

fig, axes = plt.subplots(1, 2, figsize=(18, 8))                             # Two plots side by side
for ax, emb, title in zip(axes, [cbow_embeddings, sg_embeddings],           # Loop over both models
                          ["CBOW", "Skip-Gram (Negative Sampling)"]):       # With titles
    pts = PCA(n_components=2).fit_transform(emb[ids])                       # PCA to 2-D
    ax.scatter(pts[:, 0], pts[:, 1], c=colors, s=60)                        # Coloured dots by theme
    for i, w in enumerate(highlight):                                       # Label each point
        ax.annotate(w, (pts[i, 0], pts[i, 1]), fontsize=10)                 # Word label
    ax.set_title(f"{title} embeddings (PCA)")                               # Title
    ax.grid(True)                                                           # Grid
plt.tight_layout()                                                          # Tidy layout
plt.show()                                                                  # Show

## CBOW vs Skip-Gram – summary

| Aspect | CBOW | Skip-Gram |
|---|---|---|
| Predicts | center word from context | context words from center word |
| Training examples per window | 1 | 2 × window |
| Speed | faster | slower |
| Rare words | weaker (averaged away) | better |
| Small datasets | ok | usually better |
| Output layer here | full softmax over vocab | sigmoid over 1 positive + K negatives |

## Expected observations
- Both models cluster words by theme; Skip-Gram usually separates themes a bit more clearly on small data because it creates more training pairs.

## Viva questions
1. Why negative sampling? → Avoids computing softmax over the entire vocabulary; only K+1 dot products per example.
2. Why raise frequencies to the power 0.75? → Reduces dominance of very frequent words ("the", "a") in negatives.
3. Why two embedding matrices? → A word as center and as context play different roles; using separate vectors makes training easier.
4. Why `from_logits=True`? → Model outputs raw dot products; the loss applies sigmoid internally (numerically stable).
5. Typical K? → 5–20 for small datasets, 2–5 for large ones.